# Embeddings merging strategies analysis

In [39]:
import pandas as pd
import os
import numpy as np

In [5]:
base_path = "../"

#### Load data

In [110]:
df = pd.read_csv(os.path.join(base_path, "gridsearch_merging_strategy.csv"))
df.head()

,DNABERTBACTSTRAT,NT2PHAGESTRAT,NT2BACTSTRAT,MEGADNAPHAGESTRAT,DNABERTPHAGESTRAT,MEGADNABACTSTRAT,F1Score1,Average
0,TruncateStrategy,TruncateStrategy,TruncateStrategy,TfidfStrategy,Tf4idfStrategy,MaxStrategy,0.910256,0.9102
1,TruncateStrategy,TruncateStrategy,TruncateStrategy,MaxStrategy,Tf4idfStrategy,MaxStrategy,0.920198,0.9201
2,TruncateStrategy,TruncateStrategy,TruncateStrategy,MaxStrategy,TruncateStrategy,TruncateStrategy,0.923039,0.9230
3,TruncateStrategy,TruncateStrategy,TruncateStrategy,TruncateStrategy,MaxStrategy,TruncateStrategy,0.925735,0.9257
4,TruncateStrategy,TruncateStrategy,TruncateStrategy,TfidfStrategy,TruncateStrategy,Tf4idfStrategy,0.915788,0.9157


## Analysis

In [151]:
th_high = 0.95
th_low = 0.80

In [152]:
df.sort_values(axis=0, by="Average", ascending=False, inplace=True, ignore_index=True)
df.head()

,DNABERTBACTSTRAT,NT2PHAGESTRAT,NT2BACTSTRAT,MEGADNAPHAGESTRAT,DNABERTPHAGESTRAT,MEGADNABACTSTRAT,F1Score1,Average
0,MaxStrategy,TruncateStrategy,Tf4idfStrategy,TfidfStrategy,MaxStrategy,MaxStrategy,0.953854,0.9538
1,MaxStrategy,TruncateStrategy,TfidfStrategy,MaxStrategy,Tf4idfStrategy,MaxStrategy,0.953701,0.9537
2,MaxStrategy,TKPertStrategy,TruncateStrategy,TfidfStrategy,TKPertStrategy,TruncateStrategy,0.953565,0.9535
3,MaxStrategy,TfidfStrategy,TfidfStrategy,TKPertStrategy,TruncateStrategy,TruncateStrategy,0.953387,0.9533
4,MaxStrategy,TfidfStrategy,TruncateStrategy,TfidfStrategy,MaxStrategy,MaxStrategy,0.953267,0.9532


In [153]:
df_best = df[df["Average"] >= th_high]
print(f"Number of elements selected as bests: {len(df_best)}")

df_worse = df[df["Average"] < th_low]
print(f"Number of elements selected as worse: {len(df_worse)}")

Number of elements selected as bests: 97
Number of elements selected as worse: 107


### Single column analysis

In [154]:
DF_COLUMNS = ["NT2PHAGESTRAT", "MEGADNAPHAGESTRAT", "DNABERTPHAGESTRAT", "NT2BACTSTRAT", "MEGADNABACTSTRAT", "DNABERTBACTSTRAT"]
strategies = np.unique(df[DF_COLUMNS].values)

table_best = pd.DataFrame(columns=DF_COLUMNS, index=strategies, dtype=float).fillna(0)
table_worse = pd.DataFrame(columns=DF_COLUMNS, index=strategies, dtype=float).fillna(0)

for column in DF_COLUMNS:
    table_best[column] = df_best[column].value_counts(normalize=True)
    table_worse[column] = df_worse[column].value_counts(normalize=True)

table_best.fillna(0, inplace=True)
table_worse.fillna(0, inplace=True)

In [155]:
def make_pretty(styler: pd.io.formats.style.Styler, title: str, cmap: str = "RdYlGn"):
    styler.set_caption(title)
    styler.format(precision=3)
    styler.background_gradient(axis=0, cmap=cmap)
    return styler

table_best.style.pipe(make_pretty, title=f"Best strategies (f1 >= {th_high})")

,NT2PHAGESTRAT,MEGADNAPHAGESTRAT,DNABERTPHAGESTRAT,NT2BACTSTRAT,MEGADNABACTSTRAT,DNABERTBACTSTRAT
MaxStrategy,0.134,0.247,0.216,0.299,0.485,0.773
TKPertStrategy,0.371,0.082,0.093,0.072,0.031,0.000
Tf4idfStrategy,0.155,0.268,0.278,0.072,0.062,0.000
TfidfStrategy,0.155,0.268,0.206,0.041,0.031,0.103
TruncateStrategy,0.186,0.134,0.206,0.515,0.392,0.124


In [156]:
table_worse.style.pipe(make_pretty, title=f"Worse strategies (f1 < {th_low})", cmap="RdYlGn_r")

,NT2PHAGESTRAT,MEGADNAPHAGESTRAT,DNABERTPHAGESTRAT,NT2BACTSTRAT,MEGADNABACTSTRAT,DNABERTBACTSTRAT
MaxStrategy,0.383,0.271,0.196,0.019,0.009,0.037
TKPertStrategy,0.000,0.243,0.364,0.112,0.000,0.000
Tf4idfStrategy,0.224,0.065,0.131,0.393,0.523,0.000
TfidfStrategy,0.224,0.056,0.187,0.458,0.458,0.963
TruncateStrategy,0.168,0.364,0.121,0.019,0.009,0.000
